## Problem 3 — Pruning + Static Post-Training Quantization (PTQ)

Applies **ONNX Runtime static PTQ** (INT8) on top of the 6 pruned MobileNet-v1 ONNX models
produced in Problem 1 Q3 (Strategy 2).  
Quantized models are saved to `pruned_ptq_models/`.

Source ONNX files (FP32, pruned):
```
pruned_models/prune_strat_2/complete_models/l1_{frac}_5_5_MBNv1.onnx
```
Output ONNX files (INT8, pruned + quantized):
```
pruned_ptq_models/l1_{frac}_5_5_MBNv1_int8.onnx
```

### Step 1 — Run Static PTQ via `quantize_model.py`

Calls `quantize_model.py` for each of the 6 pruned fractions.  
Calibration uses 1,000 images from the CIFAR-10 training set (handled inside `quantize_model.py`).

In [1]:
import subprocess
import sys
import os

PRUNING_FRACTIONS = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5]
SRC_DIR    = './pruned_models/prune_strat_2/complete_models'
OUTPUT_DIR = './pruned_ptq_models'

os.makedirs(OUTPUT_DIR, exist_ok=True)

for frac in PRUNING_FRACTIONS:
    src  = f'{SRC_DIR}/l1_{frac}_5_5_MBNv1.onnx'
    dst  = f'{OUTPUT_DIR}/l1_{frac}_5_5_MBNv1_int8.onnx'
    print(f'\n{"="*60}')
    print(f'  PTQ  pruning_fraction={frac}')
    print(f'  Input : {src}')
    print(f'  Output: {dst}')
    print(f'{"="*60}')

    process = subprocess.Popen(
        [
            sys.executable, 'quantize_model.py',
            '--onnx_model',   src,
            '--output_model', dst,
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end='', flush=True)
    process.wait()
    if process.returncode != 0:
        print(f'[WARNING] Process exited with code {process.returncode}')

print('\nAll PTQ experiments complete.')


  PTQ  pruning_fraction=0.05
  Input : ./pruned_models/prune_strat_2/complete_models/l1_0.05_5_5_MBNv1.onnx
  Output: ./pruned_ptq_models/l1_0.05_5_5_MBNv1_int8.onnx
C:\Users\rovez\AppData\Roaming\Python\Python311\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")
Successfully quantized model saved to ./pruned_ptq_models/l1_0.05_5_5_MBNv1_int8.onnx

  PTQ  pruning_fraction=0.1
  Input : ./pruned_models/prune_strat_2/complete_models/l1_0.1_5_5_MBNv1.onnx
  Output: ./pruned_ptq_models/l1_0.1_5_5_MBNv1_int8.onnx
C:\Users\rovez\AppData\Roaming\Python\Python311\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (D

### Step 2 — Verify Quantized Models (size + parameter count)

In [2]:
import os
import pandas as pd
import onnx

STRAT2_DIR = './pruned_models/prune_strat_2'
OUTPUT_DIR = './pruned_ptq_models'
PRUNING_FRACTIONS = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5]

def get_n_params_from_csv(frac):
    p = os.path.join(STRAT2_DIR, f'result_{frac}.csv')
    if os.path.exists(p):
        return int(pd.read_csv(p)['final_params'].iloc[0])
    return None

def get_onnx_size_mb(path):
    return os.path.getsize(path) / (1024 ** 2) if os.path.exists(path) else float('nan')

rows = []
for frac in PRUNING_FRACTIONS:
    fp32_path = f'./pruned_models/prune_strat_2/complete_models/l1_{frac}_5_5_MBNv1.onnx'
    int8_path = f'{OUTPUT_DIR}/l1_{frac}_5_5_MBNv1_int8.onnx'
    rows.append({
        'Pruning Fraction':      frac,
        'FP32 Size [MB]':        round(get_onnx_size_mb(fp32_path), 2),
        'INT8 Size [MB]':        round(get_onnx_size_mb(int8_path), 2),
        '#Parameters':           get_n_params_from_csv(frac),
        'INT8 Model Exists':     os.path.exists(int8_path),
    })

display(pd.DataFrame(rows))

,Pruning Fraction,FP32 Size [MB],INT8 Size [MB],#Parameters,INT8 Model Exists
0,0.05,11.04,2.96,2902933,True
1,0.10,9.92,2.67,2607681,True
2,0.20,7.87,2.14,2069432,True
3,0.30,6.06,1.67,1592437,True
4,0.40,4.48,1.26,1178807,True
5,0.50,3.15,0.91,828362,True


### Table 3 — Pruned + Quantized MobileNet-v1 on Raspberry Pi 3B+

Run `bash run_problem3_deploy.sh` on the Pi first, then `scp` the results back.  
Expected layout:
```
results/ptq_0.05/mobilenet_summary_metrics.csv
...
results/ptq_0.5/mobilenet_summary_metrics.csv
```

In [4]:
import os
import pandas as pd

PRUNING_FRACTIONS = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5]
RESULTS_BASE  = './results'
OUTPUT_DIR    = './pruned_ptq_models'
STRAT2_DIR    = './pruned_models/prune_strat_2'
PRUNING_ITER  = 5
FT_EPOCHS_ITER = 5  # fine-tuning epochs per pruning iteration

def load_deploy_metrics(frac):
    path = os.path.join(RESULTS_BASE, f'ptq_{frac}', 'mobilenet_summary_metrics.csv')
    if not os.path.exists(path):
        print(f'[MISSING] {path}')
        return None
    return pd.read_csv(path, header=0, names=['Metric', 'Value']).set_index('Metric')['Value']

def get_n_params_from_csv(frac):
    p = os.path.join(STRAT2_DIR, f'result_{frac}.csv')
    if os.path.exists(p):
        return int(pd.read_csv(p)['final_params'].iloc[0])
    return None

rows = []
for frac in PRUNING_FRACTIONS:
    m = load_deploy_metrics(frac)
    if m is None:
        continue

    total_images        = int(m['Total Images'])
    total_energy_j      = float(m['Total Energy (J)'])
    avg_latency_ms      = float(m['Average Inference Time per Image (ms)'])
    peak_mem_mb         = float(m['Peak RAM Memory (MB)'])
    test_accuracy       = float(m['Test Accuracy (%)'])

    energy_per_image_mj = (total_energy_j / total_images * 1000) if total_images > 0 else float('nan')

    int8_path    = f'{OUTPUT_DIR}/l1_{frac}_5_5_MBNv1_int8.onnx'
    model_size_mb = os.path.getsize(int8_path) / (1024 ** 2) if os.path.exists(int8_path) else float('nan')

    rows.append({
        'Pruning Fraction':        frac,
        '#Fine-tuning Epochs':     PRUNING_ITER * FT_EPOCHS_ITER,
        '#Parameters':             get_n_params_from_csv(frac),
        'Model Size [MB]':         round(model_size_mb, 2),
        'Max Memory Usage [MB]':   round(peak_mem_mb, 2),
        'Avg Latency/Image [ms]':  round(avg_latency_ms, 2),
        'Avg Energy/Image [mJ]':   round(energy_per_image_mj, 4),
        'Test Accuracy [%]':       round(test_accuracy, 2),
    })

table3 = pd.DataFrame(rows)
print('Table 3 — Pruned + Quantized MobileNet-v1 on Raspberry Pi 3B+\n')
display(table3.style.set_properties(**{'text-align': 'center'}).format({
    'Model Size [MB]':        '{:.2f}',
    'Max Memory Usage [MB]':  '{:.2f}',
    'Avg Latency/Image [ms]': '{:.2f}',
    'Avg Energy/Image [mJ]':  '{:.4f}',
    'Test Accuracy [%]':      '{:.2f}',
}))

# Save Table 3 to CSV
table3.to_csv('table3_pruned_ptq.csv', index=False)
print('\nTable 3 saved to table3_pruned_ptq.csv')

Table 3 — Pruned + Quantized MobileNet-v1 on Raspberry Pi 3B+



,Pruning Fraction,#Fine-tuning Epochs,#Parameters,Model Size [MB],Max Memory Usage [MB],Avg Latency/Image [ms],Avg Energy/Image [mJ],Test Accuracy [%]
0,0.050000,25,2902933,2.96,126.00,10.13,471.6710,85.81
1,0.100000,25,2607681,2.67,125.00,9.64,468.5995,85.08
2,0.200000,25,2069432,2.14,123.00,8.58,458.4180,85.59
3,0.300000,25,1592437,1.67,123.00,7.62,452.6775,84.91
4,0.400000,25,1178807,1.26,122.00,6.81,446.2084,84.32
5,0.500000,25,828362,0.91,123.00,5.51,435.6504,83.92



Table 3 saved to table3_pruned_ptq.csv


### Table 2 — Build from `table2_stats.txt` and save CSV
Parses metrics from `table2_stats.txt` and creates `table2_quantized.csv` with the same columns used by Tables 1 and 3.

In [5]:
import re
import pandas as pd

stats_path = 'table2_stats.txt'
out_csv = 'table2_quantized.csv'

with open(stats_path, 'r', encoding='utf-8') as f:
    txt = f.read()

def extract_float(pattern, text):
    m = re.search(pattern, text)
    return float(m.group(1)) if m else float('nan')

def extract_int(pattern, text):
    m = re.search(pattern, text)
    return int(m.group(1)) if m else None

# Parse Table 2 values from the text report
pruning_fraction = extract_float(r'Pruning fraction:\s*([0-9.]+)', txt)
ft_epochs = extract_int(r'#\s*Fine-tuning epochs:\s*([0-9]+)', txt)
model_size_mb = extract_float(r'Model Size \[MB\]:\s*([0-9.]+)', txt)
max_mem_mb = extract_float(r'Maximum memory usage \[MB\]:\s*([0-9.]+)', txt)
avg_latency_ms = extract_float(r'Average latency per image \[ms\]:\s*([0-9.]+)', txt)
avg_energy_mj = extract_float(r'Average energy per image \[mJ\]:\s*([0-9.]+)', txt)
test_acc = extract_float(r'Test accuracy \[%\]:\s*([0-9.]+)', txt)

# Parameters for pruning_fraction=0 are in Table 1 CSV
params = None
try:
    t1 = pd.read_csv('table1_pruned.csv')
    row0 = t1.loc[t1['Pruning Fraction'] == 0.0]
    if not row0.empty:
        params = int(row0.iloc[0]['#Parameters'])
except Exception:
    pass

# Fallback from text line: "same as unpruned in Table 1"
if params is None:
    params = float('nan')

table2 = pd.DataFrame([
    {
        'Pruning Fraction': pruning_fraction,
        '#Fine-tuning Epochs': ft_epochs,
        '#Parameters': params,
        'Model Size [MB]': model_size_mb,
        'Max Memory Usage [MB]': max_mem_mb,
        'Avg Latency/Image [ms]': avg_latency_ms,
        'Avg Energy/Image [mJ]': avg_energy_mj,
        'Test Accuracy [%]': test_acc,
    }
])

table2.to_csv(out_csv, index=False)
print(f'Table 2 saved to {out_csv}')
display(table2)

Table 2 saved to table2_quantized.csv


,Pruning Fraction,#Fine-tuning Epochs,#Parameters,Model Size [MB],Max Memory Usage [MB],Avg Latency/Image [ms],Avg Energy/Image [mJ],Test Accuracy [%]
0,0.0,100,3217226,3.3,134.0,11.78,517.09,77.61


### Compare Tables 1, 2, and 3
Loads `table1_pruned.csv`, `table2_quantized.csv`, and `table3_pruned_ptq.csv` and shows a combined comparison plus per-fraction deltas versus Table 1.

In [6]:
import pandas as pd

cols = [
    'Pruning Fraction',
    '#Fine-tuning Epochs',
    '#Parameters',
    'Model Size [MB]',
    'Max Memory Usage [MB]',
    'Avg Latency/Image [ms]',
    'Avg Energy/Image [mJ]',
    'Test Accuracy [%]',
]

t1 = pd.read_csv('table1_pruned.csv')[cols].copy()
t1['Table'] = 'Table 1 (Pruned ONNX)'

t2 = pd.read_csv('table2_quantized.csv')[cols].copy()
t2['Table'] = 'Table 2 (Unpruned PTQ)'

t3 = pd.read_csv('table3_pruned_ptq.csv')[cols].copy()
t3['Table'] = 'Table 3 (Pruned + PTQ)'

combined = pd.concat([t1, t2, t3], ignore_index=True)
combined = combined[['Table'] + cols].sort_values(['Pruning Fraction', 'Table']).reset_index(drop=True)

print('Combined comparison (Tables 1, 2, 3):')
display(combined)

# Compare Table 3 vs Table 1 for same pruning fractions
merge_13 = pd.merge(
    t1,
    t3,
    on='Pruning Fraction',
    suffixes=('_T1', '_T3')
)

for metric in ['Model Size [MB]', 'Max Memory Usage [MB]', 'Avg Latency/Image [ms]', 'Avg Energy/Image [mJ]', 'Test Accuracy [%]']:
    merge_13[f'{metric} Delta (T3-T1)'] = merge_13[f'{metric}_T3'] - merge_13[f'{metric}_T1']

compare_13 = merge_13[[
    'Pruning Fraction',
    'Model Size [MB] Delta (T3-T1)',
    'Max Memory Usage [MB] Delta (T3-T1)',
    'Avg Latency/Image [ms] Delta (T3-T1)',
    'Avg Energy/Image [mJ] Delta (T3-T1)',
    'Test Accuracy [%] Delta (T3-T1)',
]]

print('Delta comparison: Table 3 minus Table 1 (same pruning fraction):')
display(compare_13)

combined.to_csv('tables_1_2_3_comparison.csv', index=False)
compare_13.to_csv('table3_vs_table1_deltas.csv', index=False)
print('Saved: tables_1_2_3_comparison.csv and table3_vs_table1_deltas.csv')

Combined comparison (Tables 1, 2, 3):


,Table,Pruning Fraction,#Fine-tuning Epochs,#Parameters,Model Size [MB],Max Memory Usage [MB],Avg Latency/Image [ms],Avg Energy/Image [mJ],Test Accuracy [%]
0,Table 1 (Pruned ONNX),0.00,0,3217226,12.24,137.0,29.22,595.2569,77.70
1,Table 2 (Unpruned PTQ),0.00,100,3217226,3.30,134.0,11.78,517.0900,77.61
2,Table 1 (Pruned ONNX),0.05,25,2902933,11.04,145.5,29.74,589.8529,85.91
3,Table 3 (Pruned + PTQ),0.05,25,2902933,2.96,126.0,10.13,471.6710,85.81
4,Table 1 (Pruned ONNX),0.10,25,2607681,9.92,136.5,27.02,586.2085,85.03
5,Table 3 (Pruned + PTQ),0.10,25,2607681,2.67,125.0,9.64,468.5995,85.08
6,Table 1 (Pruned ONNX),0.20,25,2069432,7.87,128.0,24.04,556.3390,85.49
7,Table 3 (Pruned + PTQ),0.20,25,2069432,2.14,123.0,8.58,458.4180,85.59
8,Table 1 (Pruned ONNX),0.30,25,1592437,6.06,126.0,20.70,532.4937,85.02
9,Table 3 (Pruned + PTQ),0.30,25,1592437,1.67,123.0,7.62,452.6775,84.91


Delta comparison: Table 3 minus Table 1 (same pruning fraction):


,Pruning Fraction,Model Size [MB] Delta (T3-T1),Max Memory Usage [MB] Delta (T3-T1),Avg Latency/Image [ms] Delta (T3-T1),Avg Energy/Image [mJ] Delta (T3-T1),Test Accuracy [%] Delta (T3-T1)
0,0.05,-8.08,-19.5,-19.61,-118.1819,-0.10
1,0.10,-7.25,-11.5,-17.38,-117.6090,0.05
2,0.20,-5.73,-5.0,-15.46,-97.9210,0.10
3,0.30,-4.39,-3.0,-13.08,-79.8162,-0.11
4,0.40,-3.22,-2.0,-10.52,-63.6313,-0.02
5,0.50,-2.24,-1.0,-8.81,-56.8329,-0.01


Saved: tables_1_2_3_comparison.csv and table3_vs_table1_deltas.csv


It seems that pruned and quantized beats either just pruned or quantized. The accuracy for P+Q hoevers around the pruned accuracy although on average being slightly less. However, this disadvantage is more than made up for with the model size, memory usage, latency, and energy usage being a lot smaller than just the pruned model.